# ConstructionBots Experiment Lab

**ConstructionBots** simulates a robot "factory": a team of small mobile robots builds a LEGO model by fetching
every part and sub-assembly, carrying it (alone or in teams) to the right place, and putting it in position.
The central question is **who should carry what, and in which order**: the *task assignment*. This notebook lets
you try ten assignment algorithms (heuristic rules, exact optimization and metaheuristics), see what they
plan, watch the robots carry it out, and compare the algorithms fairly.

You do not need to write code. Change values only in the cells marked **✏️ EDIT ME**, then run the cells from top to bottom.

### Quick start (about 5 minutes)

1. Run the **Setup** cell in Section 2. Every line of its table should say **OK**.
2. Run Sections 3 and 4 unchanged. They build the Tractor with the **Genetic algorithm** and show the result,
   the robots' planned schedule, how the search improved it, and a 3-D animation.
3. Run Section 5, the **algorithm benchmark**: nine algorithms on the same problem, with comparison charts.
4. Now experiment: change `ASSIGNMENT_METHOD` in Section 3 or the list `ALGORITHMS_TO_COMPARE` in Section 5 and run again.

> **Version note.** Written for and tested with ConstructionBots 0.1.5 (this fork, branch `fork-work`), Julia 1.10.12,
> on Windows 11. Every run records the exact code version it used. See `notebooks/NOTEBOOK_USER_GUIDE.md`
> for a longer guide and `notebooks/SETUP.md` for installation.

## Contents

1. [Welcome: what is simulated](#1.-Welcome:-what-is-simulated)
2. [Setup and environment check](#2.-Setup-and-environment-check)
3. [Experiment settings (the cell you edit)](#3.-Experiment-settings)
4. [Run one experiment and see it](#4.-Run-one-experiment-and-see-it)
5. [Compare algorithms (benchmark)](#5.-Compare-algorithms-(benchmark))
6. [Repeated trials (optional)](#6.-Repeated-trials-(optional))
7. [Results files and CSV export](#7.-Results-files-and-CSV-export)
8. [Troubleshooting](#8.-Troubleshooting)
9. [Licences: what is free and what needs a licence](#9.-Licences)
10. [Known limitations](#10.-Known-limitations)

## 1. Welcome: what is simulated

**What the simulator models**

| Question | Handled by |
|---|---|
| How many robots must carry each part, and how they stand around it | transport-team configuration |
| Where each sub-assembly is built and where parts are delivered | construction-site layout ("staging plan") |
| Which robot fetches which part, in which order | **task assignment** (you choose the method) |
| How robots drive without crashing into each other or into building sites | **movement / collision avoidance** (you choose the profile) |

**What is abstracted away:** the physics of gripping and snapping LEGO pieces together, and detailed robot dynamics.
Robots are simple cylinders; parts are lifted into place by the simulator.

**What an experiment is:** one simulation of one model with one set of choices. It produces
**statistics** (most importantly the *completion time*, also called *makespan*, and a *safety record*: how close
robots came to each other) and, optionally, a 3-D **animation** you can open in a web browser.

**What you can change:** the model, the number of robots, the task-assignment algorithm (and its solver or
search budget), the movement profile, the random seed, and a time limit. Everything else stays at the repository's defaults.

### How to use this notebook

* Run a cell by clicking it and pressing **Shift + Enter**. Run cells **from top to bottom**.
* A cell is running while it shows `[*]` (Jupyter) or a spinning icon (VS Code).
* **Start fresh at any time:** use **Restart** (VS Code: *Restart* button at the top; Jupyter: *Kernel → Restart*),
  then run the cells again from the top. Nothing on disk is lost; every run is saved in its own new folder.
* The **first** experiment after a restart is slow (often 1–3 minutes) because Julia compiles the code. Later runs are fast.
* You only need to edit the cells marked **✏️ EDIT ME**: the experiment settings in Section 3 and, for a
  comparison, the comparison settings in Section 5.

## 2. Setup and environment check

Before the first use on a computer, the software must be installed once. That is described step by step in
**`notebooks/SETUP.md`** (Julia, the LDraw parts library, the Python `rvo2` module, and the Jupyter kernel).

The next cell **does not install anything**. It:

1. finds the ConstructionBots folder on this computer and activates its Julia project,
2. loads the simulator and this notebook's helper code (`notebooks/notebook_helpers.jl`),
3. checks every requirement and prints a table. Each line should say **OK**. A **WARNING** on Gurobi is normal
   if you have no Gurobi licence.

If the cell cannot find the repository, fill in `REPO_ROOT_OVERRIDE` with the folder path (keep the `raw"..."`).
If your LDraw parts library is not in `Documents\ldraw`, fill in `LDRAW_LIBRARY_OVERRIDE`.

In [ ]:
# ======================= SETUP (run once per session) =======================
# Normally leave both as `nothing`. Example of a manual path:
#   REPO_ROOT_OVERRIDE = raw"C:\Users\yourname\Documents\GitHub\ConstructionBots.jl"
REPO_ROOT_OVERRIDE = nothing
LDRAW_LIBRARY_OVERRIDE = nothing
# ============================================================================

import Pkg, TOML

function find_constructionbots_root(override)
    is_repo(d) = isfile(joinpath(d, "Project.toml")) &&
                 get(TOML.parsefile(joinpath(d, "Project.toml")), "name", "") == "ConstructionBots" &&
                 isfile(joinpath(d, "notebooks", "notebook_helpers.jl"))
    if override !== nothing
        is_repo(override) && return abspath(override)
        error("REPO_ROOT_OVERRIDE = $(repr(override)) is not a ConstructionBots.jl folder " *
              "(it must contain Project.toml and notebooks/notebook_helpers.jl).")
    end
    starts = String[pwd()]
    haskey(ENV, "CONSTRUCTIONBOTS_ROOT") && pushfirst!(starts, ENV["CONSTRUCTIONBOTS_ROOT"])
    Base.active_project() !== nothing && push!(starts, dirname(Base.active_project()))
    for start in starts
        d = abspath(start)
        while true
            is_repo(d) && return d
            parent = dirname(d)
            parent == d && break
            d = parent
        end
    end
    error("Could not find the ConstructionBots.jl folder automatically. Set REPO_ROOT_OVERRIDE " *
          "at the top of this cell to its full path, then run the cell again.")
end

REPO_ROOT = find_constructionbots_root(REPO_ROOT_OVERRIDE)
println("Repository folder: ", REPO_ROOT)
Pkg.activate(REPO_ROOT)

if !isdefined(Main, :ExperimentLab)   # load once per session; restart the kernel to reload
    println("Loading ConstructionBots and the notebook helpers (can take a minute)...")
    include(joinpath(REPO_ROOT, "notebooks", "notebook_helpers.jl"))
end
using .ExperimentLab
lab_setup(; repo_root=REPO_ROOT, ldraw_library=LDRAW_LIBRARY_OVERRIDE)

ENVIRONMENT_OK = check_environment()
if ENVIRONMENT_OK
    println("\nEverything required is ready. Continue with Section 3.")
else
    println("\nSome required item shows PROBLEM above. Follow the hint in its 'Details' column ",
            "and notebooks/SETUP.md, then restart the kernel and run this cell again.")
end

### What can I choose?

Run the next cell to list every model, assignment method, solver and movement profile with a short
explanation. Use the names **exactly as shown** (capital letters do not matter).

In [ ]:
show_models()
show_options()

## 3. Experiment settings

### The algorithms you can compare

All ten algorithms solve the same problem: they decide **which robots carry which part or sub-assembly, and
in which order**. All of them respect the model's build order and the size of every transport team, and the same
movement simulation runs afterwards. So any difference in the results comes from the assignment, and from how its
plan happens to play out in traffic.

| Family | `ASSIGNMENT_METHOD` | How it decides | Computer time (Tractor) |
|---|---|---|---|
| **Heuristic rules** (fixed rules) | `"Greedy"` | Again and again, pick the open task that a robot team can **finish earliest**, and send the robots that get there first. The repository's own method and the baseline. | well under 1 s |
| | `"Greedy with travel"` | The same as Greedy, except that each robot's finish time includes its earlier driving. This is the decoder the metaheuristics use, so it shows what they gain from the decoder alone. | well under 1 s |
| | `"Most complex first"` | Pick the open task that needs the **largest robot team** (then: whose cargo has the most LEGO parts). Big jobs first, so they do not hold up the end. | well under 1 s |
| | `"Nearest station first"` | Pick the open task whose pick-up station the free robots can **reach soonest**, judged by driving time only (not by when the robots become free). | well under 1 s |
| **Exact optimization** (MILP) | `"MILP"` | Writes the whole assignment as a **mixed-integer linear program** and lets a solver (HiGHS by default) search for the schedule with the shortest planned completion time. Stops at the time limit with the best schedule found so far, which is usually **not** the optimum: read a MILP result as "this solver for this long". | up to the time limit |
| | `"Greedy + MILP warm start"` | The same, but the solver starts from Greedy's schedule. | up to the time limit |
| **Metaheuristics** | `"Genetic algorithm"` (`"GA"`) | Keeps a **population** of candidate task orders. The better ones are combined (crossover) and slightly changed (mutation), generation after generation. | a few seconds |
| | `"Particle swarm"` (`"PSO"`) | A **swarm** of candidate task orders. Each one moves towards its own best order and the swarm's best order. | a few seconds |
| | `"Simulated annealing"` (`"SA"`) | **One** candidate order, changed a little at a time. Early on, a worse change is sometimes accepted to escape dead ends; later almost never ("cooling"). | a few seconds |
| | `"Random search"` | The **control**: the same budget of candidates, drawn at random, with no search operators. If a metaheuristic does not beat it, its operators add nothing. | a few seconds |

**How the metaheuristics work here.** A candidate solution is a list of numbers, one per transport task (so-called
*random keys*): a higher number means "assign this task earlier". A candidate is turned into a full schedule with the
same rules as Greedy (build order respected, the nearest free robots matched to the task) and is scored by its
**planned completion time**, including the robots' driving. The search starts from Greedy's task order and keeps the
best schedule it finds. Greedy's own schedule counts as a candidate too, so the **plan** is never worse than Greedy's.
A better plan does **not** guarantee a faster simulation, because the plan ignores traffic: judge the algorithms by the
simulated completion time over several seeds (Section 6). Each algorithm gets the same budget: `SEARCH_ITERATIONS × SEARCH_POPULATION` schedules
(default 25 × 16 = 400, a few seconds on the Tractor). It stops early at `TIME_LIMIT_SECONDS`. The random numbers
come from `RANDOM_SEED`, so the same settings give the same search.

### The settings

Change **one kind** of setting at a time. The *assignment method* decides **who does what**; the *movement profile*
decides **how robots drive**. If you change both and the result changes, you cannot tell which change caused it.

| Setting | What it means | When to change it |
|---|---|---|
| `MODEL` | Which LEGO model is built. Start with `"Tractor"` (20 parts, seconds). | To test a larger problem. |
| `NUMBER_OF_ROBOTS` | `nothing` = the model's recommended number (Tractor: 10), or a whole number. | To study team size. Too few robots can make the assignment impossible (the run then reports *failed*). |
| `ASSIGNMENT_METHOD` | One of the ten algorithms above (capital letters do not matter). | To compare algorithms. |
| `TIME_LIMIT_SECONDS` | The longest time the MILP solver or a metaheuristic may search per run. | Larger values may give better MILP results but take longer. Not used by the rules. |
| `SOLVER` | MILP methods only: `"HiGHS"` (free, recommended), `"GLPK"` (free), `"Gurobi"` (needs a licence). | To compare solvers. GLPK cannot be used with the warm start. |
| `SEARCH_ITERATIONS`, `SEARCH_POPULATION` | Metaheuristics only: how many iterations, and how many candidate schedules per iteration. | Larger values search longer and often find better plans. |
| `MOVEMENT_PROFILE` | `"Recommended"` (all avoidance methods on), `"No RVO2"`, `"No Tangent Bug"`, `"No Dispersion"`, `"No avoidance methods"`, or `"Custom"` (Advanced settings). | To measure what each movement method contributes. |
| `RANDOM_SEED` | A whole number that fixes the robots' starting positions (and the metaheuristics' random choices). The parts' locations stay the same. | Use different seeds to repeat an experiment fairly (Section 6). |
| `SAVE_ANIMATION` | `true` records a 3-D animation of the run, shown in Section 4. Adds a few seconds for small models. | `false` for faster runs. |
| `SAVE_RESULTS` | `true` keeps a folder with the statistics, settings and log of this run. | Keep `true` for real experiments. |

**Glossary.** *Makespan* = completion time. *MILP* = mixed-integer linear program. *RVO2* = "reciprocal velocity
obstacles": robots adjust their speed to avoid each other. *Tangent Bug* = robots steer around construction sites
in their way. *Dispersion* = a "repulsion field" that spreads robots out around busy areas.

In [ ]:
# =====================================================================
#  ✏️ EDIT ME: EXPERIMENT SETTINGS (normally the only cell you change)
# =====================================================================
MODEL                = "Tractor"             # see the "Available models" table; start small
NUMBER_OF_ROBOTS     = nothing               # nothing = model default (Tractor: 10), or e.g. 12

ASSIGNMENT_METHOD    = "Genetic algorithm"   # who carries what. Choose one:
#   heuristic rules : "Greedy", "Greedy with travel", "Most complex first", "Nearest station first"
#   exact (MILP)    : "MILP", "Greedy + MILP warm start"
#   metaheuristics  : "Genetic algorithm", "Particle swarm", "Simulated annealing", "Random search"   (or "GA", "PSO", "SA")

TIME_LIMIT_SECONDS   = 60                    # the most time the MILP solver or a metaheuristic may search
SOLVER               = "HiGHS"               # MILP only: "HiGHS" (free), "GLPK" (free), "Gurobi" (licence)
SEARCH_ITERATIONS    = 25                    # metaheuristics only: iterations (generations)
SEARCH_POPULATION    = 16                    # metaheuristics only: candidate schedules per iteration

MOVEMENT_PROFILE     = "Recommended"         # "Recommended", "No RVO2", "No Tangent Bug",
                                             # "No Dispersion", "No avoidance methods", "Custom"
RANDOM_SEED          = 1                     # whole number
SAVE_ANIMATION       = true                  # true = record a 3-D animation (shown in Section 4)
SAVE_RESULTS         = true                  # true = keep a results folder for this run

println("Settings stored. Now run the next cells.")

### Advanced settings (optional)

Most users never change these. They are listed here so nothing is hidden.

| Setting | Meaning |
|---|---|
| `CUSTOM_RVO2`, `CUSTOM_TANGENT_BUG`, `CUSTOM_DISPERSION` | Individual on/off switches, used **only** when `MOVEMENT_PROFILE = "Custom"`. |
| `SMOOTH_ANIMATION` | `true` records the animation at every simulation step (smooth, but much slower). With `false`, robots appear to jump in straight lines between events and may *look* as if they collide. |
| `ALLOW_LARGE_MODELS` | Large models (X-Wing, Passenger Plane, Star Destroyer, Kings Castle, AT-AT, Heavily Nested, Saturn V) can take from minutes to hours. They are refused unless this is `true`. |
| `MAX_SIMULATION_STEPS` | Safety limit on simulation length (simulator default 100000 steps of 1/40 s). |
| `MAX_STEPS_WITHOUT_PROGRESS` | The simulation stops if no task finishes for this many steps (simulator default 10000). Such a run is reported as *incomplete*. |
| `REUSE_PREVIOUS_RESULTS` | `true` (default): if a run with identical settings and the same code version already exists, its saved result is shown at once and nothing is simulated again (the record's status detail says so, and its computer time is 0). `false` simulates every run again, for example when you measure computer time. |
| `SIMULATE_TOP_PLANS` | Metaheuristics only. `0` (default) uses the best plan without simulating the others. Above 0, the run simulates Greedy's plan and this many of the search's best distinct plans, and keeps the fastest **in simulation**, so the simulated result is never worse than Greedy's. Costs that many extra simulations. |
| `SHOW_SIMULATOR_LOG` | `false` saves the simulator's long technical output (including the solver log) to `simulator_log.txt` in the run folder; `true` prints it here. |

In [ ]:
# ----------------------- ADVANCED SETTINGS (optional) -----------------------
CUSTOM_RVO2                = true     # used only when MOVEMENT_PROFILE = "Custom"
CUSTOM_TANGENT_BUG         = true     # used only when MOVEMENT_PROFILE = "Custom"
CUSTOM_DISPERSION          = true     # used only when MOVEMENT_PROFILE = "Custom"
SMOOTH_ANIMATION           = false
ALLOW_LARGE_MODELS         = false
MAX_SIMULATION_STEPS       = 100000   # simulator default
MAX_STEPS_WITHOUT_PROGRESS = 10000    # simulator default
REUSE_PREVIOUS_RESULTS     = true     # true = an identical earlier run is reused instead of simulated again
SIMULATE_TOP_PLANS         = 0        # metaheuristics: choose among Greedy and this many best plans by simulating them
SHOW_SIMULATOR_LOG         = false

println("Advanced settings stored.")

### Check the settings

The next cell turns your settings into an experiment description and checks it **before** anything runs.
If something is wrong it stops with a message that starts with **"Settings problem"** and lists the valid
choices. Fix the value in the settings cell and run both cells again.

In [ ]:
CONFIG = make_config(
    model = MODEL, num_robots = NUMBER_OF_ROBOTS, assignment = ASSIGNMENT_METHOD, solver = SOLVER,
    movement = MOVEMENT_PROFILE, seed = RANDOM_SEED, optimizer_time_limit_s = TIME_LIMIT_SECONDS,
    search_iterations = SEARCH_ITERATIONS, search_population = SEARCH_POPULATION,
    simulate_top_plans = SIMULATE_TOP_PLANS,
    save_animation = SAVE_ANIMATION, save_results = SAVE_RESULTS, reuse_results = REUSE_PREVIOUS_RESULTS,
    custom_rvo = CUSTOM_RVO2, custom_tangent_bug = CUSTOM_TANGENT_BUG, custom_dispersion = CUSTOM_DISPERSION,
    smooth_animation = SMOOTH_ANIMATION, allow_large_models = ALLOW_LARGE_MODELS,
    max_steps = MAX_SIMULATION_STEPS, max_iters_no_progress = MAX_STEPS_WITHOUT_PROGRESS,
)
check_settings(CONFIG);

## 4. Run one experiment and see it

Run the next cell. It prints the run's ID, then a progress bar while the robots are simulated. On the Tractor the
rules take a few seconds and the metaheuristics a few seconds more. The **first** run after a restart adds 1–3
minutes of one-time compilation. MILP methods can use the whole time limit.

The previous result is cleared **before** the new run starts, so an old result can never be mistaken for the new
one. If the run fails, its status says **FAILED** with the reason. Nothing is reported as a success unless the
simulated model was actually completed.

In [ ]:
LAST_RUN = nothing    # forget the previous result first
LAST_RUN = run_experiment(CONFIG; show_simulator_log = SHOW_SIMULATOR_LOG);

### See the results

The next cell shows everything about the run, in this order:

1. **Result table.** The main number is the **completion time (makespan)**: how long the robots took, in the
   simulated world, to finish the model. **Lower is better.** The table below explains every line.
2. **Planned schedule** (a Gantt chart). One row per robot. A grey bar means the robot is driving to its next task;
   a blue bar means its team is carrying a single part, an orange bar a sub-assembly. This is the plan the
   assignment algorithm produced. Hover over a bar to see the task and its times. Short grey bars and few gaps mean
   a tight plan.
3. **Search progress** (metaheuristics only). The best planned completion time after each iteration. Iteration 0
   is the starting point: Greedy's schedule or Greedy's task order, whichever plans better. The curve shows how the
   *plan* improved, not necessarily the simulated completion time.
4. **3-D animation** of the robots carrying out the plan. Press **Play**, drag the slider to move through time,
   drag in the picture to rotate, scroll to zoom, and press **Reset view** if you lose the model. The table above
   it checks the saved file: it says how many objects (robots, parts, transport teams) actually **move** in it.
   The animation is embedded in the notebook, so it works in Jupyter, JupyterLab and VS Code without a server.

| Measurement | Unit | Meaning |
|---|---|---|
| **Completion time (makespan)** | simulated seconds | Time until the whole model was built in the simulation. **Lower is better.** The main result. |
| Planned completion time | simulated seconds | The plan's own estimate right after assignment, ignoring traffic and avoidance manoeuvres (`OptimisticMakespan`). This is what the MILP and the metaheuristics try to minimize. The gap to the real completion time shows what movement and congestion cost. |
| Smallest gap between robots | meters | **Safety.** The closest two free robots came, surface to surface. **Negative means they overlapped.** The completion time cannot see this, so read the two together. |
| Steps with overlapping robots | simulation steps | **Safety.** How long robots overlapped (one step is 1/40 s). The second number also counts transport teams, drawn as bounding circles, so it can overstate contact. |
| Position swaps | count | **Safety.** How often two robots instantly exchanged positions to unblock a team: a teleport a real robot could not do. |
| Top speed | m/s | The fastest agent, leaving out swaps. The limit is 4 m/s. |
| Assignment time | computer seconds | Real time spent deciding who does what (`AssignmentTime` in the simulator's file). |
| Preparation / Execution time | computer seconds | Real time until the assignment is done (`PreExecutionRuntime`) / spent simulating movement (`ExecutionRuntime`). |
| Total computer time | computer seconds | Everything, measured by this notebook. |
| Valid MILP solution | yes / no | Whether the MILP solver returned an assignment (MILP methods only). |
| Solver outcome | status, gap, bound | How far the solver got (MILP methods only). `TIME_LIMIT` with a gap means the schedule is not proven optimal. |
| Search | schedules | How many candidate schedules a metaheuristic evaluated, and whether it stopped at the time limit. |
| Code version | hash, commit | Which code produced this result, so it can be traced and reproduced. |

**Simulated time is not computer time.** Simulated seconds describe the robots' world; computer seconds depend on
your computer and on whether this was the first run after a restart (compilation). A dash (—) means *not available*,
for example because the run failed or the method does not use a solver.

In [ ]:
OPEN_ANIMATION_IN_BROWSER = false     # true = also open the saved animation file in your web browser

show_visuals(LAST_RUN)                # result table, planned schedule, search progress, 3-D animation
if OPEN_ANIMATION_IN_BROWSER && LAST_RUN !== nothing && LAST_RUN[:animation_file] !== missing
    open_in_browser(LAST_RUN[:animation_file])
end;

## 5. Compare algorithms (benchmark)

A **comparison** runs several experiments one after another, changing **only the assignment algorithm** (or, in
presets C–E, one other kind of setting) and holding everything else fixed: same model, robots, seed, movement and
limits. The default is the **Custom** list in the next cell: nine algorithms covering all three families. Or
choose a preset:

| `COMPARISON_PRESET` | What is compared | Runs | Typical time (Tractor) |
|---|---|---|---|
| `"Custom"` | The algorithms in `ALGORITHMS_TO_COMPARE` | as listed | 2–4 min for the default list |
| `"A"` Baseline | Greedy + Recommended movement | 1 | under 1 min |
| `"B"` Greedy vs MILP | Greedy vs MILP vs Greedy + MILP warm start | 3 | about 2× the time limit |
| `"C"` Solver comparison | MILP with HiGHS vs GLPK vs Gurobi (skipped without a licence) | 3 | about 2× the time limit |
| `"D"` Movement ablation | Greedy with each of the 5 movement profiles | 5 | 1–2 min |
| `"E"` Scaling | Tractor with 6 / 10 / 14 robots, then X-Wing (mini) with its default 20 robots | 4 | 1–3 min |
| `"F"` Heuristic rules | Greedy vs Greedy with travel vs Most complex first vs Nearest station first | 4 | under 1 min |
| `"G"` Metaheuristics benchmark | Genetic algorithm vs Particle swarm vs Simulated annealing vs Random search (the control), with Greedy and Greedy with travel as references | 6 | 2–3 min |
| `"H"` All algorithms | all ten | 10 | about 3× the time limit |

`COMPARISON_MODEL` and `COMPARISON_ROBOTS` apply to every preset except E. Before the timed runs, `WARM_UP = true`
runs each *kind* of configuration once on the Tractor (short search, MILP limited to 30 s), untimed and unsaved,
so that one-time compilation does not distort the computer-time measurements.

**To stop a comparison early:** create an empty file named `STOP` in the batch folder printed at the start
(the current run finishes, the remaining ones are listed as *skipped*), or press the **Interrupt** button
(the current run is recorded as *interrupted*). Completed runs are always kept.

**One run per algorithm is a demonstration, not evidence.** Rankings on one seed are often reversed on the next. For
conclusions, use Section 6 (several matched seeds, with a statistical test).

> **Reading a movement ablation (preset D).** Switching avoidance methods **off** usually gives a *shorter*
> completion time (for example, Recommended 22.75 s vs No avoidance methods 14.93 s on the Tractor, seed 1).
> That is **not** an improvement: without avoidance, robots drive through each other and through building sites.
> The completion time does not penalize that, so compare movement profiles on completion time *and* on the
> safety columns (robot gap, overlap steps, swaps).

In [ ]:
# =====================================================================
#  ✏️ EDIT ME: COMPARISON SETTINGS
# =====================================================================
RUN_COMPARISON                = true        # false = skip this section (useful with "Run All")
COMPARISON_PRESET             = "Custom"    # "Custom" = the list below, or a preset "A" to "H" (table above)
ALGORITHMS_TO_COMPARE         = ["Greedy", "Greedy with travel", "Most complex first", "Nearest station first",
                                 "Genetic algorithm", "Particle swarm", "Simulated annealing", "Random search",
                                 "MILP"]    # add "Greedy + MILP warm start" if you like
COMPARISON_MODEL              = "Tractor"   # used by every preset except "E"
COMPARISON_ROBOTS             = nothing     # nothing = model default
COMPARISON_SEED               = 1
COMPARISON_TIME_LIMIT_SECONDS = 30          # per MILP / metaheuristic run
COMPARISON_SEARCH_ITERATIONS  = 25          # metaheuristics: the same budget for each of them
COMPARISON_SEARCH_POPULATION  = 16
COMPARISON_SAVE_ANIMATIONS    = false       # true = an animation for every run (slower; larger notebook)
WARM_UP                       = true

println("Comparison settings stored.")

In [ ]:
COMPARISON_RESULTS = nothing     # forget any previous comparison first
if RUN_COMPARISON
    configs, description = preset_configs(COMPARISON_PRESET; model = COMPARISON_MODEL,
        num_robots = COMPARISON_ROBOTS, seed = COMPARISON_SEED, time_limit = COMPARISON_TIME_LIMIT_SECONDS,
        save_animation = COMPARISON_SAVE_ANIMATIONS, algorithms = ALGORITHMS_TO_COMPARE,
        search_iterations = COMPARISON_SEARCH_ITERATIONS, search_population = COMPARISON_SEARCH_POPULATION)
    println(description, "\n")
    COMPARISON_RESULTS = run_batch(configs; name = "preset-" * COMPARISON_PRESET, warm_up = WARM_UP,
                                   show_simulator_log = SHOW_SIMULATOR_LOG)
else
    println("Comparison skipped (RUN_COMPARISON = false).")
end;

### Comparison charts

The next cell shows the results table and these charts (each is also saved as an `.svg` file in the batch folder):

* **Completion time in the simulation**: the main result. Each bar is one configuration, coloured by algorithm. Check
  the safety columns of the table too: a faster run with overlapping robots is not a better one.
* **Planned completion time**: what each algorithm *planned*, before traffic. Comparing the two charts shows
  whether a better plan also gave a faster build.
* **Assignment computer time**: what each algorithm cost in computer time.
* **Search progress**: the metaheuristics' improvement per iteration, with the other algorithms' planned
  completion times as dashed reference lines.
* **Planned schedules** of all runs on **one time axis**, so you can see directly which plan is tighter.

Runs that did not succeed get **no bar**; their status is written in red instead (a failed run is never drawn as zero).

In [ ]:
show_comparison(COMPARISON_RESULTS)

### Look at one compared run in detail

Shows the result table, schedule, search progress and (if `COMPARISON_SAVE_ANIMATIONS = true`) the animation of
one run of the comparison. `RUN_TO_SHOW = 1` is the first run in the table above.

In [ ]:
RUN_TO_SHOW = 4      # ✏️ position in the comparison table (1 = first row)

if COMPARISON_RESULTS === nothing
    println("Run the comparison first.")
elseif !(1 <= RUN_TO_SHOW <= length(COMPARISON_RESULTS))
    println("RUN_TO_SHOW must be a number from 1 to ", length(COMPARISON_RESULTS), ".")
else
    show_visuals(COMPARISON_RESULTS[RUN_TO_SHOW])
end

## 6. Repeated trials (this is where conclusions come from)

A single run can be lucky or unlucky. Repeating each configuration with several **random seeds** (different
robot starting positions) shows how stable a result is. Every configuration uses **the same seeds**, so the
comparison stays fair.

This section is **off by default** because it multiplies the running time (number of configurations × number
of seeds). Set `RUN_REPEATED_TRIALS = true` to use it. It reports:

* **Summary per configuration:** success rate; mean completion time of the successful runs with its 95 % confidence
  interval, standard deviation, minimum and maximum; and the safety record. Failed runs are counted, never averaged in.
* **Each configuration against Greedy, seed by seed:** the mean gain with its 95 % interval, the effect size, wins and
  losses, and a **paired sign-flip test**, corrected for several comparisons (Holm). A difference is only worth reporting
  if its interval excludes zero and the corrected p value is small.
* **Does the plan predict the simulation?** The rank correlation of planned and simulated completion time, per method.
  If it is near zero, a better plan does not mean a faster build.

**How many seeds?** With fewer than 6 matched seeds the test cannot reach p < 0.05 at all, whatever the data. Use at
least 10, and 30 for anything you intend to publish. All of this is for **one** model: a result on the Tractor does
not carry over to other models by itself. For a batch over several models, use `scripts/method_comparison.jl`.

In [ ]:
# ✏️ EDIT ME
RUN_REPEATED_TRIALS       = false
TRIALS_PRESET             = "Custom"      # "Custom" = TRIALS_ALGORITHMS, or any preset from Section 5
TRIALS_ALGORITHMS         = ["Greedy", "Greedy with travel", "Genetic algorithm", "Particle swarm",
                             "Simulated annealing", "Random search"]
TRIALS_MODEL              = "Tractor"
TRIALS_SEEDS              = 1:10          # matched seeds; 6 is the minimum for a test to reach p < 0.05
TRIALS_TIME_LIMIT_SECONDS = 30

TRIAL_RESULTS = nothing
if RUN_REPEATED_TRIALS
    base_configs, description = preset_configs(TRIALS_PRESET; model = TRIALS_MODEL,
                                               time_limit = TRIALS_TIME_LIMIT_SECONDS, algorithms = TRIALS_ALGORITHMS)
    configs = with_seeds(base_configs, TRIALS_SEEDS)
    println(description, "\nRepeated with seeds ", collect(TRIALS_SEEDS), ": ", length(configs), " runs.\n")
    TRIAL_RESULTS = run_batch(configs; name = "trials-" * TRIALS_PRESET, warm_up = true,
                              show_simulator_log = SHOW_SIMULATOR_LOG)
    show_summary(TRIAL_RESULTS; by = [:label])
    show_paired(TRIAL_RESULTS; baseline = "Greedy")
    show_plan_vs_simulation(TRIAL_RESULTS)
    folder = batch_folder(TRIAL_RESULTS)
    chart_metric(TRIAL_RESULTS, :makespan_s; title = "Completion time, mean and range over seeds",
                 ylabel = "Completion time [simulated s]", save_to = joinpath(folder, "chart_completion_time_trials.svg"))
    chart_metric(TRIAL_RESULTS, :optimistic_makespan_s; title = "Planned completion time, mean and range over seeds",
                 ylabel = "Planned completion [simulated s]", save_to = joinpath(folder, "chart_planned_trials.svg"))
else
    println("Repeated trials skipped (RUN_REPEATED_TRIALS = false).")
end

## 7. Results files and CSV export

Everything is saved under `results/notebook_lab/` inside the repository (this folder is ignored by Git):

```
results/notebook_lab/
├── all_runs.csv                     one line per saved run, appended (never overwritten)
├── runs/<run ID>/                   one new folder per run
│   ├── effective_config.toml        your settings + the exact simulator arguments + software versions
│   ├── result.toml                  the result record shown in the tables, incl. the planned schedule
│   │                                and (metaheuristics) the search progress
│   ├── simulator_log.txt            the simulator's console output (solver log, progress messages)
│   ├── error.txt                    only if the run failed: the full technical error
│   └── simulator_output/
│       ├── variable_dump/           only if the run got stuck: the simulator's debugging dump (.jld2)
│       └── <mode>_<flags>/
│           ├── <date>_stats.toml    the simulator's own statistics file; Completed = false marks a run that crashed
│           └── <date>_visualization.html  the animation, if SAVE_ANIMATION = true
└── batches/<batch ID>/
    ├── summary.csv                  the comparison table
    ├── batch_info.toml              batch settings and software versions
    └── chart_*.svg                  the charts (completion, planned, assignment time, search, schedules)
```

The CSV columns are the same everywhere; empty cells mean *not available*. Times ending in `_s` are seconds:
`makespan_s` and `optimistic_makespan_s` are **simulated** seconds; `assignment_time_s`, `pre_execution_runtime_s`,
`execution_runtime_s` and `total_wall_time_s` are **computer** seconds. `assignment_family` is "Heuristic rule",
"Exact optimization" or "Metaheuristic"; the `search_*` columns describe a metaheuristic's budget and use.
`robot_min_gap_m`, `robot_overlap_steps`, `min_gap_m`, `overlap_steps`, `position_swaps` and `max_speed_mps` are the
safety record; `solver_status`, `solver_gap` and `solver_bound` say how far a MILP solver got; `source_hash` and
`git_commit` identify the code.
When a new notebook version adds columns, the old `all_runs.csv` is kept as `all_runs_before_<date>.csv`.

The next cell exports the latest comparison to a new, time-stamped CSV file (it never overwrites a file),
reads it back to confirm it is complete, and shows how many runs are in the overall log.

In [ ]:
using Dates
records_to_export = something(TRIAL_RESULTS, COMPARISON_RESULTS,
                              LAST_RUN === nothing ? nothing : [LAST_RUN], Some(nothing))
if records_to_export === nothing
    println("Nothing to export yet.")
else
    export_path = joinpath(REPO_ROOT, "results", "notebook_lab", "exports",
                           "export_" * Dates.format(now(), "yyyymmdd-HHMMSS") * ".csv")
    export_csv(records_to_export, export_path)
    check = read_csv(export_path)
    println("Exported ", length(check), " run(s) with ", length(CSV_COLUMNS), " columns to:\n  ", export_path)
end

all_runs_file = joinpath(REPO_ROOT, "results", "notebook_lab", "all_runs.csv")
if isfile(all_runs_file)
    println("\nThe overall log ", all_runs_file, " contains ", length(read_csv(all_runs_file)), " saved run(s).")
end

## 8. Troubleshooting

| What you see | What it means | What to do |
|---|---|---|
| `Settings problem: Unknown model "..."` (or method, solver, profile) | A name is misspelled. | Copy a name from the tables printed in Section 2. |
| Tables, charts or the animation appear only as plain text | The cell output cannot show HTML (for example a plain terminal). | Open the notebook in Jupyter, JupyterLab or VS Code, restart the kernel and run from the top. |
| `Settings problem: SEARCH_ITERATIONS ...` / `SEARCH_POPULATION ...` | The metaheuristic budget is out of range. | Iterations 1–1000, population 2–500, whole numbers. |
| The result says **STOPPED AT THE TIME LIMIT** (metaheuristics) | The search used all of `TIME_LIMIT_SECONDS` before finishing its budget. | Its best plan is still used. For a full search, raise the time limit or lower the budget. |
| The saved notebook file is very large | Animations are embedded in the outputs (a few MB each). | Use *Clear All Outputs* before saving or sharing the notebook. |
| `Settings problem: ... must be a whole number` / `true or false` | A value has the wrong form, e.g. `"10"` in quotes. | Numbers and `true`/`false` go **without** quotes; names go **in** quotes. |
| `GLPK cannot be combined with "Greedy + MILP warm start"` | The simulator cannot pass a starting solution to GLPK. | Use `SOLVER = "HiGHS"`. |
| `Solver Gurobi is not usable ... no valid Gurobi licence` | Gurobi is installed as a Julia package, but this computer has no licence. | Use HiGHS, or obtain a licence (Section 9 and `SETUP.md`). |
| `... is a large model ... set ALLOW_LARGE_MODELS = true` | Protection against accidentally starting a run of minutes to hours. | Set it in Advanced settings only if you mean it. |
| Status **FAILED**, "... robots are too few: the largest transport team ... needs N" | Some parts need a team of several robots. | Set `NUMBER_OF_ROBOTS` to at least N, or use `nothing`. |
| Status **FAILED**, "no feasible solution within the time limit" | The MILP solver found no assignment in time. | Increase `TIME_LIMIT_SECONDS`, or use Greedy / the warm-start method. |
| Status **INCOMPLETE** | Robots stopped making progress (or the step limit was reached) before the model was finished. Completion time is left empty on purpose. | Try another movement profile or seed; see the run's `simulator_log.txt`. |
| Environment check: `Python module rvo2: PROBLEM` | The simulator needs the Python RVO2 library for **every** run, even with RVO2 off. | Follow `SETUP.md`, step 3, then restart the kernel. |
| Environment check: `LDraw Parts Library: PROBLEM` | The part shapes are missing. | Install the library (`SETUP.md`, step 2) or set `LDRAW_LIBRARY_OVERRIDE`. |
| `UndefVarError: ... not defined` | A cell was run before the cells above it. | Restart the kernel and run from the top. |
| The first run takes minutes | One-time compilation after every restart. | Normal. Later runs are fast. |
| The animation shows the finished model but nothing moves; the check says **0 moving objects** | The file was saved by an older version of this code, which recorded no movement. | Run the experiment again. |
| The animation frame stays empty or says the page cannot be reached | Only for very large animations, which are served by the notebook's Julia session instead of being embedded; the session was restarted. | Run the cell again, or open the saved file directly (`OPEN_ANIMATION_IN_BROWSER = true`). |
| The animation shows robots passing through each other | With `SMOOTH_ANIMATION = false` the animation only updates when tasks finish, so robots are drawn on straight lines between updates. With "No avoidance methods" they really do not avoid each other. | Use `SMOOTH_ANIMATION = true` (slower) to see real avoidance. |

**A finished simulation does not prove that no collisions happened.** The avoidance modules reduce conflicts but do
not guarantee collision-free motion, so check the safety record of the run (smallest gap, overlap steps, swaps).
It covers free robots and formed transport teams, not parts lying on the ground.

## 9. Licences

**Everything needed for the default experiments is free.** The heuristic rules and the metaheuristics need no
solver at all (they are part of the repository's Julia code); the MILP
methods work with **HiGHS** (MIT licence) or **GLPK** (GPL-3.0), both installed automatically with the project.

| Component | Licence | Notes |
|---|---|---|
| ConstructionBots.jl | MIT (`LICENSE`, © 2021 Kyle Brown and contributors) | Free to use, modify and share with the copyright notice. Please cite the paper (see `README.md`). |
| This notebook and its helper files | Part of the repository; same MIT terms as the repository unless your lab decides otherwise. | |
| HiGHS solver | MIT | Free, including commercial use. |
| GLPK solver (and GLPK.jl) | GNU GPL-3.0 | Free to use; redistributing software that *includes* GLPK has GPL obligations. |
| ECOS (used internally for the site layout) | GPL-3.0 (ECOS.jl wrapper: MIT) | Installed automatically. |
| JuMP | MPL-2.0 | |
| RVO2 library (sisl/Python-RVO2 fork) | Apache-2.0 (© University of North Carolina at Chapel Hill) | Installed separately (`SETUP.md`). |
| LDraw Parts Library | CC BY 2.0 / CC BY 4.0 (see `CAreadme.txt` in the library) | Download it yourself from ldraw.org; do not commit it to the repository. If you share images or parts, give attribution to LDraw.org. LEGO® is a trademark of the LEGO Group, which does not endorse this project. |
| **Gurobi** solver | **Commercial** (Gurobi EULA) | **Optional.** The `Gurobi` Julia package installs the solver program, but it will not run without a **licence of your own**. |

**About Gurobi licences.** Free academic licences are offered to students, faculty and staff of accredited
degree-granting institutions for non-commercial teaching and research. A *Named-User* academic licence is
personal and tied to one computer: having one does **not** give other lab members the right to use Gurobi, and
licence files must never be put in this notebook or in the repository. Each user should check their own
eligibility and obtain their own licence (see `SETUP.md`, "Optional: Gurobi", and https://www.gurobi.com/academia/).
Always confirm the current terms on Gurobi's website; they can change.

This notebook uses **no web services or AI APIs**; every experiment runs locally.

## 10. Known limitations

* **RVO2 is always required.** Even with RVO2 switched off, the simulator creates an RVO2 simulation object.
* **Dispersion only acts together with RVO2.** In this version of the code, the dispersion (potential-field)
  velocity adjustment is applied inside the RVO2 branch of the controller. With RVO2 off, the dispersion setting
  has little or no effect on how robots move.
* **Results older than this version are not comparable.** Results used to change whenever the package was recompiled.
  That is fixed, so numbers from before the fix (with no `source_hash`) cannot be compared with current ones.
  A change to `src/` or to a package version can still change results on purpose: `source_hash` shows when it did,
  and earlier results are not reused then. Results on another operating system were not tested and may differ slightly.
* **Metaheuristics search the task order only, and they minimize the plan.** Robot pairing and the build order follow
  Greedy's rules, so they explore a part of all possible schedules. They minimize the *planned* completion time. The
  simulated one includes traffic, can rank the algorithms differently, and is what counts (see Section 6).
* **The simulator abstracts physics.** Velocity changes are instant unless you pass `max_acceleration` to
  `run_lego_demo`, robots can swap positions instantly, the top speed is 4 m/s whatever the model's scale, and parts are
  loaded faster than robots drive on small models. So completion times are not comparable across models and do not
  predict real-robot times. No part of this has been calibrated against real hardware.
* **Experiments vary only the robots' start positions** (and the metaheuristics' random numbers) by default. The parts'
  locations are fixed unless you use `scripts/method_comparison.jl --placements`.
* **MILP results depend on the time limit.** With the default limits the MILP solver usually stops at the time
  limit without proving its answer optimal; its schedule can be *worse* than Greedy's. Report the time limit with
  any MILP result.
* **Computer-time measurements** vary between computers and include compilation on the first run of each kind
  after a restart (the `includes_one_time_compilation` column marks these runs).
* **Determinism.** Runs of the rules and the metaheuristics with the same code, settings and seed give the same
  result, also after a recompile (a metaheuristic only if it finishes its budget before the time limit). The seed
  controls the robots' starting positions. **MILP runs that stop at the time limit are not exactly reproducible**: how far the solver gets in
  the allowed time depends on the computer's speed and load. On the test computer, three identical
  MILP + HiGHS runs (Tractor, seed 1, 30 s) gave completion times of 55.9, 50.3 and 49.8 s. Use several seeds
  and repetitions, and report the time limit.
* **Saving an animation** starts MeshCat's viewer server on `127.0.0.1` (this computer only) during the run; it
  is closed when the run ends. The saved HTML file contains the viewer program and all scene data and works
  without any server or internet connection. `show_animation` embeds it in the cell output (files up to
  30 MB), so it shows in any notebook front end and survives a kernel restart; larger files are served
  read-only from the notebook's Julia session (`127.0.0.1`).
* **Safety is recorded, not enforced.** Brief overlaps can occur even with all avoidance methods on, and the completion
  time does not penalize them. Report the safety record with any completion time.
* Large models (hundreds to thousands of parts) can take from minutes to hours and much memory.